# 077 — LoRA, QLoRA y adaptación eficiente

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

- **Problema**: full fine-tuning con Adam cuesta ~16 bytes/parámetro (7B → ~112 GB)
  y produce una copia del modelo por tarea.
- **LoRA**: congela W₀ y aprende ΔW = (α/r)·B·A con A ∈ ℝ^{r×k}, B ∈ ℝ^{d×r}
  (B inicializada en 0). Entrena r·(d+k) parámetros en vez de d·k; fusionable en
  inferencia (latencia extra cero).
- **Conteo clave**: W de 768×768 → 589 824 parámetros; LoRA r=8 → 12 288 (~2,1 %).
- **QLoRA**: base congelado en NF4 (4 bits, cuantiles de una normal) + doble
  cuantización + optimizadores paginados; adaptadores y gradientes en BF16.
  Permite ajustar 65B en una GPU de 48 GB.
- **Hiperparámetros**: r = 8–16 típico, α = r o 2r, LR ~1e-4, aplicar a todas las
  lineales suele rendir más.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("neural", seed=77)
show(result)


## Reflexión

1. ¿Por qué B se inicializa en cero y A gaussiana, y qué pasaría al inicio del
   entrenamiento si ambas fueran gaussianas?
2. Parámetros LoRA crecen como r·(d+k) y la matriz completa como d·k: ¿por qué esto
   hace a LoRA *relativamente* más eficiente cuanto más grande es el modelo?
3. ¿En qué escenario de producto elegirías servir 20 adaptadores sin fusionar sobre
   un base compartido en vez de 20 modelos fusionados, y qué pagas a cambio?